# 03. Chọn đặc trưng cơ sở

Nạp các phiên bản dữ liệu đã xử l
So sánh `RMSE`, `MAE`, `R2` để chọn baseline feature set.
Lưu bảng kết quả


In [ ]:
import sys
import json
import time
from pathlib import Path

current_path = Path.cwd().resolve()
candidate_paths = [current_path, *current_path.parents]
PROJECT_ROOT = next(
    (
        path for path in candidate_paths
        if (path / 'data').exists() and (path / 'src').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Không thể tự động xác định thư mục dự án AbaloneAge. '
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.tree import DecisionTreeRegressor

from src.data.split_data import attach_target, split_features_target
from src.models.evaluate import build_kfold, evaluate_regression_model, evaluate_regression_metrics


## 1. Khai báo đường dẫn và helper


In [3]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'
MODELS_DIR = PROJECT_ROOT / 'outputs' / 'models'

METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'Rings'
RANDOM_STATE = 42

def save_table(df, output_stem):
    df.to_csv(output_stem.with_suffix('.csv'), index=False)
    output_stem.with_suffix('.json').write_text(
        json.dumps(df.to_dict(orient='records'), indent=2),
        encoding='utf-8',
    )

def load_processed_datasets():
    split_files = {
        'train': {
            'encoded_only': PROCESSED_DIR / 'abalone_train_encoded.csv',
            'standard_scaled': PROCESSED_DIR / 'abalone_train_standard.csv',
            'robust_log_scaled': PROCESSED_DIR / 'abalone_train_robust_log.csv',
        },
        'test': {
            'encoded_only': PROCESSED_DIR / 'abalone_test_encoded.csv',
            'standard_scaled': PROCESSED_DIR / 'abalone_test_standard.csv',
            'robust_log_scaled': PROCESSED_DIR / 'abalone_test_robust_log.csv',
        },
    }

    has_explicit_splits = all(path.exists() for version in split_files.values() for path in version.values())
    datasets = {'train': {}, 'test': {}}

    if has_explicit_splits:
        for split_name, version_map in split_files.items():
            for version_name, csv_path in version_map.items():
                df = pd.read_csv(csv_path)
                datasets[split_name][version_name] = (
                    df.drop(columns=[TARGET_COL]),
                    df[TARGET_COL],
                )
        return datasets

    encoded_df = pd.read_csv(PROCESSED_DIR / 'abalone_processed_no_scaling.csv')
    scaled_df = pd.read_csv(PROCESSED_DIR / 'abalone_processed_scaled.csv')

    x_train_encoded, x_test_encoded, y_train_encoded, y_test_encoded = split_features_target(
        encoded_df, TARGET_COL, test_size=0.3, random_state=RANDOM_STATE
    )
    x_train_scaled, x_test_scaled, y_train_scaled, y_test_scaled = split_features_target(
        scaled_df, TARGET_COL, test_size=0.3, random_state=RANDOM_STATE
    )

    train_encoded_df = attach_target(x_train_encoded, y_train_encoded, TARGET_COL)
    test_encoded_df = attach_target(x_test_encoded, y_test_encoded, TARGET_COL)
    train_scaled_df = attach_target(x_train_scaled, y_train_scaled, TARGET_COL)
    test_scaled_df = attach_target(x_test_scaled, y_test_scaled, TARGET_COL)

    datasets['train']['encoded_only'] = (train_encoded_df.drop(columns=[TARGET_COL]), train_encoded_df[TARGET_COL])
    datasets['test']['encoded_only'] = (test_encoded_df.drop(columns=[TARGET_COL]), test_encoded_df[TARGET_COL])
    datasets['train']['standard_scaled'] = (train_scaled_df.drop(columns=[TARGET_COL]), train_scaled_df[TARGET_COL])
    datasets['test']['standard_scaled'] = (test_scaled_df.drop(columns=[TARGET_COL]), test_scaled_df[TARGET_COL])
    datasets['train']['robust_log_scaled'] = (train_scaled_df.drop(columns=[TARGET_COL]), train_scaled_df[TARGET_COL])
    datasets['test']['robust_log_scaled'] = (test_scaled_df.drop(columns=[TARGET_COL]), test_scaled_df[TARGET_COL])
    return datasets

def run_cv_metrics(model, x_data, y_data):
    cv = build_kfold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    y_oof_pred = cross_val_predict(model, x_data, y_data, cv=cv, n_jobs=1)
    return evaluate_regression_metrics(y_data, y_oof_pred)

def evaluate_experiment(model, x_train, y_train, x_test, y_test):
    cv_start = time.perf_counter()
    cv_metrics = run_cv_metrics(clone(model), x_train, y_train)
    cv_time_sec = time.perf_counter() - cv_start

    fitted_model = clone(model)
    fit_start = time.perf_counter()
    fitted_model.fit(x_train, y_train)
    fit_time_sec = time.perf_counter() - fit_start

    y_pred = fitted_model.predict(x_test)
    test_metrics = evaluate_regression_model(y_test, y_pred)
    return cv_metrics, test_metrics, cv_time_sec, fit_time_sec


## 2. Nạp dữ liệu đã tiền xử lí



In [4]:
datasets = load_processed_datasets()
for split_name, split_data in datasets.items():
    print(split_name.upper())
    for version_name, (x_data, y_data) in split_data.items():
        print(f'  - {version_name}: X={x_data.shape}, y={y_data.shape}')


TRAIN
  - encoded_only: X=(2923, 10), y=(2923,)
  - standard_scaled: X=(2923, 10), y=(2923,)
  - robust_log_scaled: X=(2923, 10), y=(2923,)
TEST
  - encoded_only: X=(1254, 10), y=(1254,)
  - standard_scaled: X=(1254, 10), y=(1254,)
  - robust_log_scaled: X=(1254, 10), y=(1254,)


## 3. Chạy so sánh bộ đặc trưng cơ sở


In [5]:
feature_plan = [
    {'experiment_name': 'linear_regression_standard', 'family': 'linear_probe', 'dataset_variant': 'standard_scaled', 'model_name': 'LinearRegression', 'model': LinearRegression()},
    {'experiment_name': 'linear_regression_robust', 'family': 'linear_probe', 'dataset_variant': 'robust_log_scaled', 'model_name': 'LinearRegression', 'model': LinearRegression()},
    {'experiment_name': 'ridge_regression_standard', 'family': 'linear_probe', 'dataset_variant': 'standard_scaled', 'model_name': 'RidgeRegression', 'model': Ridge()},
    {'experiment_name': 'ridge_regression_robust', 'family': 'linear_probe', 'dataset_variant': 'robust_log_scaled', 'model_name': 'RidgeRegression', 'model': Ridge()},
    {'experiment_name': 'svr_standard', 'family': 'linear_probe', 'dataset_variant': 'standard_scaled', 'model_name': 'SVR', 'model': SVR()},
    {'experiment_name': 'svr_robust', 'family': 'linear_probe', 'dataset_variant': 'robust_log_scaled', 'model_name': 'SVR', 'model': SVR()},
    {'experiment_name': 'sgd_standard', 'family': 'linear_probe', 'dataset_variant': 'standard_scaled', 'model_name': 'SGDRegressor', 'model': SGDRegressor(random_state=RANDOM_STATE, max_iter=2000, tol=1e-3)},
    {'experiment_name': 'sgd_robust', 'family': 'linear_probe', 'dataset_variant': 'robust_log_scaled', 'model_name': 'SGDRegressor', 'model': SGDRegressor(random_state=RANDOM_STATE, max_iter=2000, tol=1e-3)},
    {'experiment_name': 'tree_encoded', 'family': 'tree_probe', 'dataset_variant': 'encoded_only', 'model_name': 'DecisionTreeRegressor', 'model': DecisionTreeRegressor(random_state=RANDOM_STATE)},
    {'experiment_name': 'tree_standard', 'family': 'tree_probe', 'dataset_variant': 'standard_scaled', 'model_name': 'DecisionTreeRegressor', 'model': DecisionTreeRegressor(random_state=RANDOM_STATE)},
    {'experiment_name': 'forest_encoded', 'family': 'tree_probe', 'dataset_variant': 'encoded_only', 'model_name': 'RandomForestRegressor', 'model': RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=200, n_jobs=1)},
    {'experiment_name': 'forest_standard', 'family': 'tree_probe', 'dataset_variant': 'standard_scaled', 'model_name': 'RandomForestRegressor', 'model': RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=200, n_jobs=1)},
    {'experiment_name': 'boosting_encoded', 'family': 'tree_probe', 'dataset_variant': 'encoded_only', 'model_name': 'GradientBoostingRegressor', 'model': GradientBoostingRegressor(random_state=RANDOM_STATE)},
    {'experiment_name': 'boosting_standard', 'family': 'tree_probe', 'dataset_variant': 'standard_scaled', 'model_name': 'GradientBoostingRegressor', 'model': GradientBoostingRegressor(random_state=RANDOM_STATE)},
]

records = []
for item in feature_plan:
    x_train, y_train = datasets['train'][item['dataset_variant']]
    x_test, y_test = datasets['test'][item['dataset_variant']]
    cv_metrics, test_metrics, cv_time_sec, fit_time_sec = evaluate_experiment(
        item['model'], x_train, y_train, x_test, y_test
    )
    records.append({
        'experiment_name': item['experiment_name'],
        'family': item['family'],
        'model_name': item['model_name'],
        'dataset_variant': item['dataset_variant'],
        'cv_rmse': cv_metrics['rmse'],
        'cv_mae': cv_metrics['mae'],
        'test_rmse': test_metrics['rmse'],
        'test_mae': test_metrics['mae'],
        'test_r2': test_metrics['r2'],
        'cv_time_sec': cv_time_sec,
        'fit_time_sec': fit_time_sec,
    })

feature_results = pd.DataFrame(records).sort_values(['family', 'test_rmse', 'test_mae']).reset_index(drop=True)
display(feature_results)


,experiment_name,family,model_name,dataset_variant,cv_rmse,cv_mae,test_rmse,test_mae,test_r2,cv_time_sec,fit_time_sec
0,svr_standard,linear_probe,SVR,standard_scaled,2.180578,1.482348,2.160870,1.491397,0.540171,2.772083,0.444493
1,ridge_regression_robust,linear_probe,RidgeRegression,robust_log_scaled,2.188713,1.566469,2.166889,1.557175,0.537606,0.035930,0.002654
2,svr_robust,linear_probe,SVR,robust_log_scaled,2.197660,1.490488,2.174461,1.495234,0.534368,2.243187,0.502098
3,linear_regression_robust,linear_probe,LinearRegression,robust_log_scaled,2.184263,1.568446,2.177008,1.559670,0.533277,0.027781,0.002092
4,sgd_robust,linear_probe,SGDRegressor,robust_log_scaled,2.247976,1.587754,2.177165,1.582212,0.533210,0.147709,0.029883
5,ridge_regression_standard,linear_probe,RidgeRegression,standard_scaled,2.262505,1.602004,2.185328,1.582906,0.529703,0.035355,0.002843
6,linear_regression_standard,linear_probe,LinearRegression,standard_scaled,2.262193,1.602625,2.187416,1.583238,0.528804,0.228294,0.003542
7,sgd_standard,linear_probe,SGDRegressor,standard_scaled,2.290091,1.611383,2.193412,1.585546,0.526217,0.073206,0.018710
8,boosting_encoded,tree_probe,GradientBoostingRegressor,encoded_only,2.171701,1.530359,2.179970,1.539953,0.532006,3.562551,0.712683
9,boosting_standard,tree_probe,GradientBoostingRegressor,standard_scaled,2.171010,1.529943,2.181155,1.542060,0.531497,2.947726,0.808107


## 4. Tổng hợp theo từng nhóm mô hình


In [6]:
feature_summary = (
    feature_results.groupby(['family', 'dataset_variant'], as_index=False)
    .agg(
        mean_test_rmse=('test_rmse', 'mean'),
        mean_test_mae=('test_mae', 'mean'),
        mean_test_r2=('test_r2', 'mean'),
        experiment_count=('experiment_name', 'count'),
    )
    .sort_values(['family', 'mean_test_rmse', 'mean_test_mae'])
    .reset_index(drop=True)
)
display(feature_summary)


,family,dataset_variant,mean_test_rmse,mean_test_mae,mean_test_r2,experiment_count
0,linear_probe,robust_log_scaled,2.173881,1.548573,0.534615,4
1,linear_probe,standard_scaled,2.181756,1.560772,0.531224,4
2,tree_probe,standard_scaled,2.446193,1.713968,0.397249,3
3,tree_probe,encoded_only,2.450719,1.717598,0.394415,3


## 5. Chọn bộ đặc trưng cơ sở cho bước tiếp theo


In [7]:
linear_choice = feature_summary.query("family == 'linear_probe'").sort_values('mean_test_rmse').head(1)
tree_choice = feature_summary.query("family == 'tree_probe'").sort_values('mean_test_rmse').head(1)

display(linear_choice)
display(tree_choice)


,family,dataset_variant,mean_test_rmse,mean_test_mae,mean_test_r2,experiment_count
0,linear_probe,robust_log_scaled,2.173881,1.548573,0.534615,4


,family,dataset_variant,mean_test_rmse,mean_test_mae,mean_test_r2,experiment_count
2,tree_probe,standard_scaled,2.446193,1.713968,0.397249,3


## 6. Lưu kết quả


In [8]:
save_table(feature_results, METRICS_DIR / 'feature_selection_baseline_results')
save_table(feature_summary, METRICS_DIR / 'feature_selection_baseline_summary')
print('đã lưu kết quả feature baseline vào outputs/metrics/.')


đã lưu kết quả feature baseline vào outputs/metrics/.
